# IAIA-BL Emory External Validation File Selection

Creates data selection for IAIA-BL External validation by generating symlinks to ROI images. The ROI Images must have already been processed by `extra_rois_embed.py`. That dataset is merged with the EMBED dataset to create the final selected links.

In [ ]:
import pandas as pd
import numpy as np
import re
import math
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.image as img
import matplotlib.patches as patches
import os
# import imageio
# import aiofiles
import asyncio
import re
# import cv2
import logging
import urllib
import torch
import torch.nn.functional as F
from tqdm import tqdm
from IPython.display import display
from ast import literal_eval
from pathlib import Path
import time
import pickle
import PIL

import shutil
import sys

from random import sample

from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.patches import Rectangle

tqdm.pandas()
basepath = '/redacted/path/to/embed'

logger = logging.getLogger(__name__)

device = torch.device(f'cuda') if torch.cuda.is_available() else torch.device('cpu')

# Setup Utility Functions
These are from our utility function library, inlined here for readability.

Please do not fork these directly. Instead, if you want to start using these, let's discuss how we can incorporate the changes back into our main library.

In [ ]:
def safe_mass_id(row):
    if 'massshape' in row:
        has_mass = (row['massshape'] in ['G', 'R', 'O', 'X', 'N', 'Y', 'D', 'L']) or \
        (row['massmargin'] in ['D', 'U', 'M', 'I', 'S']) or \
        (row['massdens'] in ['+', '-', '='])
    elif 'massshape_scr' in row:
        has_mass = (row['massshape_scr'] in ['G', 'R', 'O', 'X', 'N', 'Y', 'D', 'L']) or \
        (row['massshape_diag'] in ['G', 'R', 'O', 'X', 'N', 'Y', 'D', 'L']) or \
        (row['massmargin_scr'] in ['D', 'U', 'M', 'I', 'S']) or \
        (row['massmargin_diag'] in ['D', 'U', 'M', 'I', 'S']) or \
        (row['massdens_scr'] in ['+', '-', '=']) or \
        (row['massdens_diag'] in ['+', '-', '='])
    else:
        return False
    
    if has_mass:
        spec = '_scr' if 'acc_anon_scr' in row else ''
        return f"{row[f'acc_anon{spec}']}{row[f'side']}{row[f'numfind{spec}']}"
    else:
        return np.nan
    

def stats(df, patient_column=None):

    if patient_column is not None:
        patient_column = df[patient_column]
    elif 'empi_anon' in df.index.names:
        patient_column = df.index.to_frame()["empi_anon"]
    else:
        patient_column = df["empi_anon"]

    stats_dict = {
        "rows": df.shape[0],
        "columns": df.shape[1],
        "patients": patient_column.nunique(),
    }

    if 'png_path' in df.columns:
        stats_dict['images (path)'] = df['png_path'].nunique()
    if 'png_filename' in df.columns:
        stats_dict['image (file)'] = df['png_filename'].nunique()
    
    if 'acc_anon_scr' in df.columns:
        stats_dict["screen"] = df["acc_anon_scr"].nunique()
    if 'acc_anon_diag' in df.columns:
        stats_dict["diagnostic"] = df["acc_anon_diag"].nunique()
    if 'acc_anon' in df.columns:
       stats_dict["exams"] = df["acc_anon"].nunique()

    masses = df.apply(safe_mass_id, axis=1).nunique()
    stats_dict['masses'] = masses
   
    # pathology = df['malignancy_label'].value_counts()
    # acquisition = df['acquistion_type'].value_counts()

    margin_col = 'massmargin' if 'massmargin_gold' not in df.columns else 'massmargin_gold'

    if margin_col in df.columns:
        margin = df[margin_col].value_counts()
        stats_dict['circumscribed'] = margin['D'] if 'D' in margin else 0
        stats_dict['indistinct'] = margin['I'] if 'I' in margin else 0
        stats_dict['spiculated'] = margin['S'] if 'S' in margin else 0

    return pd.DataFrame(stats_dict, index=[0])

def lead_with_columns(df, lead_columns):
    """
    Rearranges dataframe column order by pulling a few columns to the front but keeping the order for other columns
    """
    post_columns = [col for col in df.columns if col not in lead_columns]
    return df[lead_columns + post_columns]

def col_search(df, pattern, ignore_case=True):
    """
    Find columns that match a pattern
    """
    flag = re.NOFLAG if ignore_case == False else re.I
    return [c for c in df.columns if re.match(pattern, c, flags = flag)]

def metadata_for(df_legend, cols):
    """
    Generate a dataframe with the metadata for a particular column in the embed dataset.
    """
    return df_legend[df_legend['Header in export']
          .isin(cols)] \
          .sort_values('Header in export')

def basic_count(df, cols):
    if 'acc_anon' in df.columns:
        return df[cols + ['empi_anon',  'acc_anon']].groupby(cols, dropna=False).count()

## Data Manipulation

In [ ]:
KEY_COLUMNS_MAG = ['index', 'empi_anon', 'acc_anon', 'side', 'numfind', 'desc', 'asses']
KEY_COLUMNS_MET = ['Unnamed: 0', 'empi_anon', 'acc_anon', 'ImageLateralityFinal', 'ViewPosition', 'FinalImageType', 'AcquisitionTime']
CONTEXT_COLUMNS = ['cohort_num', 'study_date_anon']

def load_met(file_path=os.path.join(basepath, 'metadata_all_cohort_with_ROI_HITI.csv'), 
             scope='min', extra_cols=[], cohorts=None, nrows=None):
    
    force_string_cols = ['CollimatorShape', '0_ProcedureCodeSequence_CodeValue', 'DerivationDescription',
                         'CommentsOnRadiationDose', 'DetectorDescription', 'WindowCenter', 'WindowWidth']
    
    categorical_cols = ['AcquisitionDeviceProcessingCode', 'AcquisitionDeviceProcessingDescription',
                        'DetectorConfiguration', 'FieldOfViewShape', 'CollimatorShape',
                        'DetectorActiveShape', 'ExposureStatus', 'VOILUTFunction'
                        '0_IconImageSequence_PhotometricInterpretation']
    
    tuple_cols = ['FieldOfViewDimensions', 'DetectorActiveDimensions', 'DetectorElementPhysicalSize',
                  'DetectorElementSpacing', 'WindowCenterWidthExplanation']
    
    date_cols = ['study_date_anon']
    
    forced_types = {col: 'str' for col in force_string_cols}
    forced_types.update({col: 'category' for col in categorical_cols})
    forced_types.update({col: 'str' for col in tuple_cols})
    
    if scope == 'min':
        usecols = (KEY_COLUMNS_MET + CONTEXT_COLUMNS + 
                    ['png_path', 'png_filename', 'num_roi', 'ROI_coords', 'match_level', 'spot_mag'] + extra_cols)
    
    elif scope == 'full':
        usecols = None
    
    else:
        return NotImplemented(f'{scope} is not a valid column scope')
    
    df_metadata = pd.read_csv(file_path, dtype=forced_types, parse_dates=date_cols, usecols=usecols, nrows=nrows)
    
    if cohorts:
        df_metadata = df_metadata.loc[df_metadata['cohort_num'].isin(cohorts)]

    return df_metadata.pipe(lead_with_columns, KEY_COLUMNS_MET + CONTEXT_COLUMNS)

__birads_map = {
    'A': 0,
    'N': 1,
    'B': 2,   
    'P': 3,
    'S': 4,
    'M': 5,
    'K': 6,
    'X': 'X'
}
    # BIRADS 0: A – Additional evaluation
    # BIRADS 1: N – Negative
    # BIRADS 2: B - Benign
    # BIRADS 3: P – Probably benign
    # BIRADS 4: S – Suspicious
    # BIRADS 5: M- Highly suggestive of malignancy
    # BIRADS 6: K - Known biopsy proven

def translate_birads(hiti_asses):
    return __birads_map[hiti_asses]


def load_mag(file_path=os.path.join(basepath, 'magview_all_cohorts_anon_HITI.csv'), 
             scope = 'min', extra_cols=[], cohorts=None):
    # columns that have inconsistent value types - default casting doesn't work properly
    inconsistent_columns = ['case', 'biopsite', 'bcomp', 'path7', 'path8', 'path9', 'path10', 'hgrade',
                            'tnmpt', 'tnmpn', 'tnmm', 'tnmdesc', 'stage', 'bdepth','focality',
                            'specinteg', 'specembed', 'her2', 'fish', 'extracap', 'methodevl',
                            'eic', 'first_3_zip']
    type_overrides = {col: 'str' for col in inconsistent_columns}

    categories = ['massshape', 'massmargin', 'massdens', 'calcfind', 'calcdistri', 'otherfind', 
                  'implanfind', 'side', 'location', 'depth', 'distance', 'asses', 'recc', 'proccode',
                  'vtype','tissueden', 'MARITAL_STATUS_DESC']
    type_overrides.update({col: 'category' for col in categories})

    date_cols = {'study_date_anon','sdate_anon','procdate_anon','pdate_anon'}
    
    if scope == 'min':
        usecols = (KEY_COLUMNS_MAG + CONTEXT_COLUMNS + 
            ['massshape', 'massmargin', 'massdens', 'otherfind', 'path_group', 'path_severity'] + extra_cols)
        date_cols = date_cols.intersection(usecols)
        
    elif scope == 'full':
        usecols = None
        
    else:
        return NotImplemented(f'{scope} is not a valid column scope')
    
    df_mag =  pd.read_csv(file_path, dtype=type_overrides, header=0,
                          parse_dates=list(date_cols), usecols=usecols)
    
    df_mag['scr_or_diag'] = df_mag['desc'].apply(lambda desc: 'S' if 'screen' in desc.lower() else 'D').astype("category")
    df_mag['birads'] = df_mag['asses'].apply(translate_birads)

    if cohorts:
        df_mag = df_mag.loc[df_mag['cohort_num'].isin(cohorts)]

    return df_mag.pipe(lead_with_columns, KEY_COLUMNS_MAG + CONTEXT_COLUMNS)


def select__goldscr(def_met, image_type='2D'):
    """
    Selects the 'gold' screening images for exams according to our selection criteria so that each of the 4 views only has one image.
    """
    def_met_goldscreen = def_met[(def_met['ViewPosition'].isin(['MLO', 'CC'])) 
                              & (def_met['ImageLateralityFinal'].isin(['L', 'R']))
                              & (def_met['FinalImageType'] == image_type)
                              & (def_met['spot_mag'].isna())
                              & (def_met['scr_or_diag'] == 'S')]
    
    if def_met.index.name == 'acc_anon':
        def_met['acc_anon'] = def_met.index

    def_met_goldscreen = def_met_goldscreen.sort_values(['AcquisitionTime'], ascending=False)
    def_met_goldscreen = def_met_goldscreen.drop_duplicates(['ViewPosition', 'ImageLateralityFinal'])

    return def_met_goldscreen

## Data Selection
Here we select our the datasets that meet the IAIA-BL input criteria to use for External Validation.

In order, we make the following selections:

1. Images labeled with ROI
1. Mass Pathology Confirmed through Biopsy
1. Mass Margins of Circumscribed, Indistinct, or Spiculated

In [ ]:
# Initial the Datasets
df_mag = load_mag(extra_cols=['calcfind', 'total_L_find', 'total_R_find'])
display(df_mag.head(4))

df_mag['has_calc'] = ~df_mag['calcfind'].isna()
df_mag['has_other'] = ~df_mag['otherfind'].isna()

df_met = load_met(extra_cols=['ImageType',
'FilterType',
'WindowCenter',
'WindowWidth',
'Manufacturer'])

display(df_met.head(4))

In [ ]:
df_mag.pipe(stats)

In [ ]:
df_met.pipe(stats)

In [ ]:
# Screen and Diagnostic Subsets
df_mag_scr = df_mag.loc[df_mag['scr_or_diag'] == 'S']

display(df_mag_scr.pipe(stats))

df_mag_diag = df_mag.loc[df_mag['scr_or_diag'] == 'D']

display(df_mag_diag.pipe(stats))

In [ ]:
# Top line Numbers
df_magXmet = df_met.merge(df_mag, left_on=['acc_anon', 'empi_anon', 'ImageLateralityFinal'], 
                    right_on=['acc_anon', 'empi_anon', 'side'], 
                    suffixes=('_met', '_mag'),
                    how='outer')

df_magXmet.pipe(stats)

In [ ]:
#scr_diag don't make sense here, but just keeping the convention so the stats function works
df_magXmet_rediag = df_magXmet.merge(df_mag_diag, on=['empi_anon', 'side'], suffixes=['_scr', '_diag'], how='left')
df_magXmet_rediag.pipe(stats)

### 1. Images labeled with ROIs

In [ ]:
# Number of Screening Exams
scr_exam_ids = set(df_mag_scr['acc_anon'])
df_met_scr = df_met.loc[df_met.acc_anon.isin(scr_exam_ids)]
display(len(scr_exam_ids))
display(df_met_scr.pipe(stats))
# notice the loss of 14k 'screening' exams that have no images

In [ ]:
# extra ROIs
from collections import namedtuple
from ast import literal_eval as make_tuple

roi = namedtuple('roi', ['top', 'bottom', 'dist'])

def parse_roi(row):

    pre_tuples = make_tuple(row['ROI_coords'])
    if len(pre_tuples) == 0:
        return []
    
    rois = []
    tuples = []
    for maybe_tuple in pre_tuples:
        if len(maybe_tuple) == 0:
            print('empty tuple:', maybe_tuple)
            continue

        if type(maybe_tuple[0]) == list:
            for tup in maybe_tuple:
                tuples.append(tup)
        else:
            tuples.append(maybe_tuple)
    
    for t in tuples:
        
        try:
            y_dist = t[2] - t[0]
            x_dist = t[3] - t[1]

            bottom = (t[3], t[2])
            top= (t[1], t[0])

            rois.append(roi(top, bottom, (x_dist, y_dist)))
        except Exception as e:
            print(row, t)
            print(row['ROI_coords'])
            raise e
        
    return rois

df_met_scr['roi_tuples'] = df_met_scr.progress_apply(parse_roi, axis=1)
df_met_scr['roi_tuples_len'] = df_met_scr['roi_tuples'].progress_apply(lambda x: len(x))
df_met_scr.groupby(['roi_tuples_len']).count()

In [ ]:
# Get Exams with ROIs
df_met_scr_roi = df_met_scr.loc[df_met_scr.roi_tuples_len > 0]
display(df_met_scr_roi.pipe(stats))
scr_exam_wroi_ids = set(df_met_scr_roi.acc_anon)
len(scr_exam_wroi_ids)

In [ ]:
# df that will delay filtering to get full stats on masses, images, and patients
df_4stat_met_scr = df_met_scr.copy()
df_4stat_mag_scr = df_mag_scr.copy()

In [ ]:
df_met_scr_roi_2d = df_met_scr_roi.loc[df_met_scr_roi['FinalImageType'] == '2D']
df_met_scr_roi_2d.pipe(stats)

In [ ]:
df_mag_scr.side.value_counts()

In [ ]:
# Expand the dataset so that side='B' is treated as L and R
df_mag_scr_sideB = df_mag_scr.loc[df_mag_scr.side == 'B']
df_mag_scr_sideLR = df_mag_scr.loc[df_mag_scr.side != 'B']

display('LR Only', df_mag_scr_sideLR.pipe(stats))

for side in ['L', 'R']:
    df_mag_scr_side = df_mag_scr_sideB.copy()
    df_mag_scr_side['side'] = side
    display('with ' + side, df_mag_scr_side.pipe(stats))
    df_mag_scr_sideLR = pd.concat([df_mag_scr_sideLR, df_mag_scr_side])

df_mag_scr_B = df_mag_scr_sideLR
display('All', df_mag_scr_B.pipe(stats))

In [ ]:
# Expand the dataset so that side='B' is treated as L and R
df_4stat_mag_scr_sideB = df_4stat_mag_scr.loc[df_4stat_mag_scr.side == 'B']
df_4stat_mag_scr_sideLR = df_4stat_mag_scr.loc[df_4stat_mag_scr.side != 'B']

display('LR Only', df_4stat_mag_scr_sideLR.pipe(stats))

for side in ['L', 'R']:
    __df_mag_scr_side = df_mag_scr_sideB.copy()
    __df_mag_scr_side['side'] = side
    display('with ' + side, __df_mag_scr_side.pipe(stats))
    df_4stats_mag_scr_sideLR = pd.concat([df_4stat_mag_scr_sideLR, __df_mag_scr_side])

df_4stat_mag_scr_B = df_4stat_mag_scr_sideLR
display('All', df_mag_scr_B.pipe(stats))

In [ ]:
df_magXmet_scr_Wroi_2d = df_met_scr_roi_2d.merge(df_mag_scr_B,
                                      left_on=['acc_anon', 'empi_anon', 'ImageLateralityFinal'], 
                                      right_on=['acc_anon', 'empi_anon', 'side'], 
                                      suffixes=('_met', '_mag'))
df_magXmet_scr_Wroi_2d['study_date_anon'] = df_magXmet_scr_Wroi_2d['study_date_anon_mag']

# This is about 300 less exams than have ROIs
display(df_magXmet_scr_Wroi_2d.pipe(stats))

In [ ]:
df_4stat_magXmet_scr = df_4stat_met_scr.merge(df_4stat_mag_scr_B,
                                      left_on=['acc_anon', 'empi_anon', 'ImageLateralityFinal'], 
                                      right_on=['acc_anon', 'empi_anon', 'side'], 
                                      suffixes=('_met', '_mag'))
df_4stat_magXmet_scr['study_date_anon'] = df_4stat_magXmet_scr['study_date_anon_mag']

# This is about 300 less exams than have ROIs
display(df_4stat_magXmet_scr.pipe(stats))

In [ ]:
# df_magXmet_scr_Wroi is the main result of this section
df_magXmet_scr_Wroi_2d.head(3)

### 2. Mass Pathology Confirmed through Biopsy
To get this, we need to
1. Join screening exams with follow-up diagnostic exams to find where there is a confirmed pathology
2. Choose a gold pathology from the two

In [ ]:
df_magXmet_scrXdiag_Wroi_2d = df_magXmet_scr_Wroi_2d.merge(df_mag_diag, on=['empi_anon', 'side'], suffixes=['_scr', '_diag'])
display(df_magXmet_scrXdiag_Wroi_2d.columns, df_magXmet_scrXdiag_Wroi_2d.pipe(stats))

In [ ]:
# This generates a filtering function so you can select a different month ranges to compare, if you like
def filter_forward_date_for(source_col, filter_col, months_range=6):
    def filter_by_date_range(row):
        start_date = row[source_col] 
        end_date = row[source_col] + pd.DateOffset(months=months_range)
        return (row[filter_col] >= start_date) & (row[filter_col] <= end_date)
    return filter_by_date_range

for months in [6]:
    proc_date_filter = filter_forward_date_for('study_date_anon_scr','study_date_anon_diag', months_range=months)
    df_magXmet_scrXdiag_Wroi_2d[f'screen_{months}_before_diag'] = df_magXmet_scrXdiag_Wroi_2d.apply(proc_date_filter, axis=1)
    
df_magXmet_scrXdiag_WroiW6monfollow_2d = df_magXmet_scrXdiag_Wroi_2d.loc[df_magXmet_scrXdiag_Wroi_2d['screen_6_before_diag'] == True]
df_magXmet_scrXdiag_Wroi_2d = df_magXmet_scrXdiag_Wroi_2d.pipe(lead_with_columns, ['empi_anon', 'acc_anon_scr', 'study_date_anon_scr', 'acc_anon_diag', 'study_date_anon_diag', 'path_severity_scr', 'path_severity_diag'])

display(df_magXmet_scrXdiag_WroiW6monfollow_2d.pipe(stats))
df_magXmet_scrXdiag_WroiW6monfollow_2d.head(3)

In [ ]:
def gold_margin(row):
    if type(row['massmargin_diag']) != float:
        return row['massmargin_diag']
    else:
        return row['massmargin_scr']

df_magXmet_scrXdiag_WroiW6monfollow_2d['worst_path_severity'] = df_magXmet_scrXdiag_WroiW6monfollow_2d[['path_severity_scr', 'path_severity_diag']].min(axis=1)
df_magXmet_scrXdiag_WroiW6monfollow_2d['massmargin_gold'] = df_magXmet_scrXdiag_WroiW6monfollow_2d.apply(gold_margin, axis=1)

df_magXmet_scrXdiag_WroiW6monfollow_2d_sorted = df_magXmet_scrXdiag_WroiW6monfollow_2d.sort_values(['empi_anon', 'acc_anon_scr', 'worst_path_severity', 'study_date_anon_diag'], ascending=True)
df_magXmet_scrXdiag_WroiW6monfollow_2d_sorted.groupby(['worst_path_severity', 'massmargin_gold']).count()[['empi_anon']]

### Stats Version

In [ ]:
df_4stat_magXmet_scrXdiag = df_4stat_magXmet_scr.merge(df_mag_diag, on=['empi_anon', 'side'], suffixes=['_scr', '_diag'], how='left')
df_4stat_magXmet_scrXdiag.pipe(stats)

In [ ]:
df_4stat_magXmet_scrXdiag = df_4stat_magXmet_scrXdiag[(df_4stat_magXmet_scrXdiag['FinalImageType'] == '2D') & (df_4stat_magXmet_scrXdiag['roi_tuples_len'] > 0)].pipe(stats)

### End: Extra Stats

In [ ]:
__df = df_magXmet_scrXdiag_WroiW6monfollow_2d_sorted

df_magXmet_scrXdiag_WroiW6monfollowWpath_2d_sorted = __df[~__df['worst_path_severity'].isna()]

df_magXmet_scrXdiag_WroiW6monfollowWpath_2d_sorted.pipe(stats)

In [ ]:
__df = df_magXmet_scrXdiag_WroiW6monfollowWpath_2d_sorted

__df = __df.sort_values(['worst_path_severity'])

# Remove sides with multiple findings
dfs = {}
for side in ['L', 'R']:
    df_side = __df.loc[__df['side'] == side]

    df_side = df_side.loc[(df_side[f'total_{side}_find_scr'] == 1) & (df_side[f'total_{side}_find_diag'] == 1)] 
    dfs[side] = df_side

__df = pd.concat((dfs['L'], dfs['R']))


# limit ROIs
__df = __df[__df['roi_tuples_len'] == 1]


# standard screening views
__df = __df[__df['ViewPosition'].isin(['CC', 'MLO'])]

__df = __df[__df['otherfind_scr'].isna() & __df['otherfind_diag'].isna()]
__df = __df[__df['calcfind_scr'].isna() & __df['calcfind_diag'].isna()]
# __df = __df[~__df['worst_path_severity'].isna()]

df_magXmet_scrXdiag_WroiW6monfollowWpathWonefind_2d_sorted = __df

df_magXmet_scrXdiag_WroiW6monfollowWpathWonefind_2d_sorted.pipe(stats)

### 3. Mass Margins of Circumscribed, Indistinct, or Spiculated
With the gold screening to pathology mappings, we simply select images that have the correct mass margins

In [ ]:
__df.columns

In [ ]:
# Adding columns "mass", "asymmetry", "arch_distortion" and "calc" as a summary of imaging findings contained in the 
#other columns. This will be coded as 1 = present; 0 = absent
__df = df_magXmet_scrXdiag_WroiW6monfollowWpathWonefind_2d_sorted

# Instantiate lists for the four finding type -  mass, asymmetry, architectural distortion and calcification
# Default value set to 0. 
mass_list = [0]*__df.shape[0]
asymmetry_list = [0]*__df.shape[0]
arch_destortion_list = [0]*__df.shape[0]
calc_list = [0]*__df.shape[0]


# Architectural Distortion is defined as: 'massshape' ['Q', 'A']
# Asymmetry is defined as: 'massshape' in ['T', 'B', 'S', 'F', 'V']
# Mass is defined as: 'massshape' in ['G', 'R', 'O', 'X', 'N', 'Y', 'D', 'L']
#       or 'massmargin' in ['D', 'U', 'M', 'I', 'S']
#       or 'massdens' in ['+', '-', '=']
# Calcification: defined as presence of any non-zero or non-null value in "calcdistri", "calcfind" or "calcnumber"

#iterate through rows and assign values to the lists based on above rules
def get_mass(row):
    if (row['massshape_scr'] in ['G', 'R', 'O', 'X', 'N', 'Y', 'D', 'L']) or \
        (row['massshape_diag'] in ['G', 'R', 'O', 'X', 'N', 'Y', 'D', 'L']) or \
        (row['massmargin_scr'] in ['D', 'U', 'M', 'I', 'S']) or \
        (row['massmargin_diag'] in ['D', 'U', 'M', 'I', 'S']) or \
        (row['massdens_scr'] in ['+', '-', '=']) or \
        (row['massdens_diag'] in ['+', '-', '=']):
        return 1
    else:
        return 0 
'''        
    if row['massshape'] in ['T', 'B', 'S', 'F', 'V']:
        asymmetry_list[ind] = 1

    if row['massshape']in ['Q', 'A']:
        arch_destortion_list[ind] = 1
        
    if (row['calcdistri'] is not np.nan) or (row['calcfind'] is not np.nan) or (row['calcnumber'] != 0):
        calc_list[ind] = 1        '''

# Append the final image findings columns to the dataframe        
__df['has_mass'] = __df.apply(get_mass, axis=1)

__df.sample(3)

In [ ]:
__df[__df['has_mass'] == 1].pipe(stats)
df_magXmet_scrXdiag_WroiW6monfollowWpathWonefindWmass = __df[__df['has_mass'] == 1]

In [ ]:
iaiabl_margins = {
    'D': 'Circumscribed',
    'I': 'Indistinct',
    'S': 'Spiculated'
}

all_margins = {
    'D': 'Circumscribed',
    'I': 'Indistinct',
    'S': 'Spiculated',
    'U': 'Obscured',
    'M': 'Microlobulated',
}

In [ ]:
__df = df_magXmet_scrXdiag_WroiW6monfollowWpathWonefind_2d_sorted

__df = __df[__df['massmargin_gold'].isin(iaiabl_margins)]

df_magXmet_scrXdiag_WroiW6monfollowWpathWonefindWiablmargins_2d_sorted = __df

df_magXmet_scrXdiag_WroiW6monfollowWpathWonefindWiablmargins_2d_sorted.pipe(stats)

In [ ]:
__df = df_magXmet_scrXdiag_WroiW6monfollow_2d_sorted

df_magXmet_scrXdiag_WroiW6monfollowWiaiablmargin_2d = __df.loc[__df['massmargin_diag'].isin(iaiabl_margins.keys()) | __df['massmargin_scr'].isin(iaiabl_margins.keys())]

df_magXmet_scrXdiag_WroiW6monfollowWiaiablmargin_2d.groupby(['worst_path_severity']).count()[['empi_anon']]

# Link Creation Code
This was done to separate ROI preprocessing from final dataset selection because of resource constraints.
`marginless` refers to the selected ROIs that do not have margins but do have pathology labels and can be used for context window tuning.
However, because the model expects data to be structured by margin, the resulting directories have to have images randomly distributed.

In [ ]:
df_roi_files = pd.read_csv('/redacted/path/to/image-metadata.csv') # output of extract_rois_emory.py
df_roi_files.head(3)

In [ ]:
# data moved to shared directories

def update_path(df, source_dir='/redacted/path/to/rois/'):
    df['old_path'] = df['path']
    df['path'] = (source_dir + df['path'].str.split('/').str[-1])

update_path(df_roi_files)
df_roi_files['path'][0]

In [ ]:
df_roi_files_marginless = pd.read_csv('/redacted/path/to/image-metadata.csv')
update_path(df_roi_files_marginless, source_dir='/redacted/path/to/rois-marginless')

df_roi_files_marginless['path'][0]

In [ ]:
df_magXmet_scrXdiag_WroiW6monfollowWpathWonefindWmass['png_filename'].value_counts()

In [ ]:
df_magXmet_scrXdiag_WroiW6monfollowWiaiablmargin_2d_Xroifiles = df_magXmet_scrXdiag_WroiW6monfollowWiaiablmargin_2d.merge(df_roi_files, left_on='png_filename', right_on='original_image')

In [ ]:
# Not losing any exams!
df_magXmet_scrXdiag_WroiW6monfollowWpathWonefindWmass.pipe(stats).pipe(display)

df_magXmet_scrXdiag_WroiW6monfollowWpathWonefindWmass_Xroifiles = df_magXmet_scrXdiag_WroiW6monfollowWpathWonefindWmass.merge(df_roi_files_marginless, left_on='png_filename', right_on='original_image')
df_magXmet_scrXdiag_WroiW6monfollowWpathWonefindWmass_Xroifiles.pipe(stats).pipe(display)


In [ ]:
def create_symbolic_links(file_groups, root_path):
    exists, new, no_target = [], [], []
    for group, file_paths in file_groups:
        # Create the directory with the group name if it doesn't exist
        group_path = root_path / str(group)
        group_path.mkdir(parents=True)
        # Iterate through the file paths and create symbolic links
        for file_path in file_paths['path']:
            # Get the file name from the file path
            file_name = os.path.basename(file_path)

            file_path_path = Path(file_path)
            symlink_path = group_path / file_name

            if not file_path_path.exists():
                no_target.append((symlink_path, file_path_path))
                continue

            # Create a symbolic link in the group directory
            
            if symlink_path.exists():
                print(f'symlink already exists, {symlink_path}')
                exists.append((symlink_path, file_path_path))
                continue

            os.symlink(file_path, symlink_path, target_is_directory=False)
            new.append((symlink_path, file_path_path))

    return {
        "new": new,
        "exists": exists,
        "no_target": no_target
    }

# Example dictionary
file_groups = pd.DataFrame.from_dict({
    "path": ["path/to/file1", "path/to/file2", "path/to/file3", "path/to/file4"],
    "group": ['group1', 'group1', 'group2', 'group2']
})
test_path = Path('./test')
if test_path.exists() and test_path.is_dir():
    for sub in test_path.iterdir():
        sub.rmdir()
    test_path.rmdir()
create_symbolic_links(file_groups.groupby(['group']), test_path)

In [ ]:
# This assumes the images have already had the ROIs extracted in copy_imgs.py

def symlink_margin_groups(df_subgroup, name, margin_col='massmargin_gold', target_dir='/redacted/target/path'):
    
    target_dir = Path(target_dir) / name
    
    df_subgroup = df_subgroup.copy()
    df_subgroup['massmargin_full'] = df_subgroup[margin_col].apply(lambda x: all_margins[x] if type(x) != float else 'X')
    
    links = create_symbolic_links(df_subgroup.groupby(['massmargin_full'], dropna=False), target_dir)
    df_subgroup.to_csv(target_dir/'index.csv')
    return links

test_path = Path('test')
if test_path.exists():
    for subdir in test_path.iterdir():
        for file in subdir.iterdir():
            os.remove(file)
        subdir.rmdir()
    test_path.rmdir()

symlink_margin_groups(df_magXmet_scrXdiag_WroiW6monfollowWpathWonefindWmass_Xroifiles.sample(3), 'test_copy', target_dir='test')


# Link Generation

## Test Set
- No multiple findings for the same side
- No multiple ROIs
- Standard Screening Views

In [ ]:
__df = df_magXmet_scrXdiag_WroiW6monfollowWiaiablmargin_2d_Xroifiles

__df = __df.sort_values(['worst_path_severity'])

# Remove sides with multiple findings
dfs = {}
for side in ['L', 'R']:
    df_side = __df.loc[__df['side'] == side]

    df_side = df_side.loc[(df_side[f'total_{side}_find_scr'] == 1) & (df_side[f'total_{side}_find_diag'] == 1)] 
    dfs[side] = df_side

__df = pd.concat((dfs['L'], dfs['R']))


# limit ROIs
__df = __df[__df['roi_tuples_len'] == 1]


# standard screening views
__df = __df[__df['ViewPosition'].isin(['CC', 'MLO'])]

__df = __df[__df['otherfind_scr'].isna() & __df['otherfind_diag'].isna()]
__df = __df[__df['calcfind_scr'].isna() & __df['calcfind_diag'].isna()]
__df = __df[~__df['worst_path_severity'].isna()]

df_test = __df.sort_values(['study_date_anon_diag']).drop_duplicates(['png_path'])

df_test.pipe(stats).pipe(display)
display(df_test.groupby(['worst_path_severity', 'massmargin_gold']).count())

# Uncomment to regenerate
# symlink_margin_groups(df_oneroi, 'one_roi')

In [ ]:
df_test.groupby(['massmargin_gold']).count()

## Validation Set
(for tuning context window)
- No multiple findings for the same side
- No multiple ROIs
- Standard Screening Views
- Mass margins not required

In [ ]:
__df = df_magXmet_scrXdiag_WroiW6monfollowWpathWonefindWmass_Xroifiles

__df = __df.sort_values(['worst_path_severity'])

# Remove sides with multiple findings
dfs = {}
for side in ['L', 'R']:
    df_side = __df.loc[__df['side'] == side]

    df_side = df_side.loc[(df_side[f'total_{side}_find_scr'] == 1) & (df_side[f'total_{side}_find_diag'] == 1)] 
    dfs[side] = df_side

__df = pd.concat((dfs['L'], dfs['R']))

# limit ROIs
__df = __df[__df['roi_tuples_len'] == 1]

# standard screening views
__df = __df[__df['ViewPosition'].isin(['CC', 'MLO'])]

__df = __df[__df['otherfind_scr'].isna() & __df['otherfind_diag'].isna()]
__df = __df[__df['calcfind_scr'].isna() & __df['calcfind_diag'].isna()]
__df = __df[~__df['worst_path_severity'].isna()]

df_validation = __df.sort_values(['study_date_anon_diag']).drop_duplicates(['png_path'])

df_validation.pipe(stats).pipe(display)
display(df_validation.groupby(['worst_path_severity', 'massmargin_gold']).count())

# Uncomment to regenerate
# Note that, because there are no margin labels, this uses made up margin labels to satisfy the directory structure
# symlink_margin_groups(df_validation, 'validation_marginless')